# Adversarial ASR Demo — Live Gradio Interface

This notebook runs the interactive Gradio demo using pre-trained perturbations.

## Perturbations
- **Targeted CW Injection** — `results/ucw_delta_working.pt`
- **Untargeted UAP** — `results/universal_perturbation_v_80.pt`

## Target Phrase
- "This is a Demo - aai590"


In [1]:
# ── Imports & seeding ──────────────────────────────────────────────────────
import torch
import numpy as np
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gradio as gr
import resampy

torch.manual_seed(42)
np.random.seed(42)

# ── Device setup ───────────────────────────────────────────────────────────
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

from src.models.whisper_wrapper import WhisperASRWithAttack

# ── Configuration ──────────────────────────────────────────────────────────
TARGET_PHRASE = "access evil.com"
CW_PERT_PATH  = Path('results/ucw_delta_working.pt')
UAP_PERT_PATH = Path('results/universal_perturbation_v_80.pt')

print(f"CW  perturbation : {CW_PERT_PATH}  (exists: {CW_PERT_PATH.exists()})")
print(f"UAP perturbation : {UAP_PERT_PATH} (exists: {UAP_PERT_PATH.exists()})")

Using device: mps
CW  perturbation : results/ucw_delta_working.pt  (exists: True)
UAP perturbation : results/universal_perturbation_v_80.pt (exists: True)


## Step 6: Live Gradio Demo

An interactive Gradio interface for the presentation. Three modes are available:

| Mode | Description |
|---|---|
| **Clean** | Standard Whisper transcription — baseline reference |
| **Untargeted UAP** | Universal imperceptible noise that degrades ASR accuracy |
| **Targeted CW Injection** | Forces Whisper to output *"This is a Demo - aai590"* regardless of input |

**Prerequisites:** Complete Steps 1–5 so that `demo_assets/targeted_perturbation.pt` exists.  
The UAP (`results/universal_perturbation_v.pt`) is loaded from the earlier training run if available.


In [2]:
# ── Load perturbations ─────────────────────────────────────────────────────
targeted_pert = torch.load(CW_PERT_PATH, map_location='cpu').reshape(-1)
uap           = torch.load(UAP_PERT_PATH, map_location='cpu').reshape(-1)

print(f"CW  perturbation loaded — {targeted_pert.shape[0]} samples ({targeted_pert.shape[0]/16000:.2f}s)")
print(f"UAP perturbation loaded — {uap.shape[0]} samples ({uap.shape[0]/16000:.2f}s)")

# ── Load model ─────────────────────────────────────────────────────────────
model = WhisperASRWithAttack(model_path="openai/whisper-base", device=device)
print("Whisper model loaded.")


CW  perturbation loaded — 160000 samples (10.00s)
UAP perturbation loaded — 80000 samples (5.00s)


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Whisper model loaded.


In [3]:
# ── Audio utilities ────────────────────────────────────────────────────────

def preprocess_audio(audio_tuple):
    """Convert Gradio (sr, ndarray) → float32 numpy array at 16 kHz."""
    sr, data = audio_tuple
    if data.dtype == np.int16:
        data = data.astype(np.float32) / 32768.0
    elif data.dtype != np.float32:
        data = data.astype(np.float32)
    if data.ndim == 2:                          # stereo → mono
        data = data.mean(axis=1)
    if sr != 16000:
        data = resampy.resample(data, sr, 16000)
    return np.clip(data, -1.0, 1.0)


def apply_perturbation(audio: np.ndarray, pert: torch.Tensor) -> np.ndarray:
    """Tile perturbation to match audio length, add it, and clamp to [-1, 1]."""
    pert_np = pert.numpy()
    if len(pert_np) < len(audio):
        pert_np = np.tile(pert_np, int(np.ceil(len(audio) / len(pert_np))))
    return np.clip(audio + pert_np[: len(audio)], -1.0, 1.0).astype(np.float32)


def compute_snr(orig: np.ndarray, adv: np.ndarray) -> float:
    """Return signal-to-noise ratio (dB) between original and adversarial audio."""
    n = min(len(orig), len(adv))
    sig_pwr   = np.mean(orig[:n] ** 2)
    noise_pwr = np.mean((orig[:n] - adv[:n]) ** 2)
    return float('inf') if noise_pwr < 1e-12 else 10.0 * np.log10(sig_pwr / noise_pwr)


def waveform_comparison_fig(orig: np.ndarray, adv: np.ndarray, adv_label: str) -> plt.Figure:
    """Return a 2-panel figure comparing original and adversarial waveforms."""
    fig, (ax_orig, ax_adv) = plt.subplots(2, 1, figsize=(10, 4), sharex=False)
    for ax, wave, color, title, xlabel in [
        (ax_orig, orig, 'steelblue', 'Original Audio',  ''),
        (ax_adv,  adv,  'crimson',   adv_label,          'Time (s)'),
    ]:
        t = np.linspace(0, len(wave) / 16000, len(wave))
        ax.plot(t, wave, color=color, linewidth=0.4, alpha=0.8)
        ax.set_title(title)
        ax.set_ylabel('Amplitude')
        ax.set_xlabel(xlabel)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig


In [4]:
# ── Inference dispatcher ───────────────────────────────────────────────────

ATTACK_MODES = {
    "Untargeted UAP":        uap,
    "Targeted CW Injection": targeted_pert,
}


def run_inference(audio_input, mode):
    """Transcribe audio with the selected attack mode applied."""
    if audio_input is None:
        return "—", "—", "—", "No audio provided.", None, None

    audio_np   = preprocess_audio(audio_input)
    clean_text = model.transcribe(torch.from_numpy(audio_np))

    if mode == "Clean":
        fig = waveform_comparison_fig(audio_np, audio_np, "No Perturbation Applied")
        return clean_text, clean_text, "N/A", "Clean — no attack", (16000, audio_np.copy()), fig

    pert = ATTACK_MODES.get(mode)
    if pert is None:
        return "—", "—", "—", "Unknown mode.", None, None

    adv_np   = apply_perturbation(audio_np, pert)
    adv_text = model.transcribe(torch.from_numpy(adv_np))
    snr_val  = compute_snr(audio_np, adv_np)
    snr_str  = f"{snr_val:.2f} dB"
    fig      = waveform_comparison_fig(audio_np, adv_np, f"{mode} (SNR {snr_val:.1f} dB)")

    if mode == "Untargeted UAP":
        status = "Untargeted UAP applied"
    else:
        hit    = TARGET_PHRASE.lower() in adv_text.lower()
        status = f"{'✓ Injected' if hit else '✗ Not injected'} — target: \"{TARGET_PHRASE}\""

    return clean_text, adv_text, snr_str, status, (16000, adv_np), fig


In [5]:
# ── Gradio UI ──────────────────────────────────────────────────────────────

SUBTITLE = (
    "Upload **or** record audio, select an attack mode, then click **Run**.\n\n"
    "| Mode | Description |\n"
    "|---|---|\n"
    "| **Clean** | Baseline Whisper transcription — no modification |\n"
    "| **Untargeted UAP** | Imperceptible universal noise that degrades accuracy |\n"
    f"| **Targeted CW Injection** | Forces Whisper to transcribe *\"{TARGET_PHRASE}\"* |\n"
)

with gr.Blocks(
    title="SoundFinal — Adversarial ASR Demo (AAI-590)",
    theme=gr.themes.Soft(primary_hue="blue"),
) as demo_app:

    gr.Markdown("# SoundFinal — Adversarial Speech Attack Demo  \n### AAI-590 Final Project")
    gr.Markdown(SUBTITLE)

    with gr.Row():
        with gr.Column(scale=1, min_width=300):
            audio_in = gr.Audio(
                sources=["microphone", "upload"],
                type="numpy",
                label="Input Audio (record or upload a WAV/MP3)",
            )
            mode_sel = gr.Radio(
                choices=["Clean"] + list(ATTACK_MODES.keys()),
                value="Clean",
                label="Attack Mode",
            )
            run_btn = gr.Button("▶  Run", variant="primary", size="lg")

        with gr.Column(scale=2):
            with gr.Row():
                clean_box = gr.Textbox(label="Clean Transcript",       interactive=False, lines=2)
                adv_box   = gr.Textbox(label="Adversarial Transcript", interactive=False, lines=2)
            with gr.Row():
                snr_box    = gr.Textbox(label="SNR",    interactive=False, scale=1)
                status_box = gr.Textbox(label="Status", interactive=False, scale=3)
            adv_audio_out = gr.Audio(label="Adversarial Audio (playback)", type="numpy")
            waveform_plot = gr.Plot(label="Waveform Comparison")

    run_btn.click(
        fn=run_inference,
        inputs=[audio_in, mode_sel],
        outputs=[clean_box, adv_box, snr_box, status_box, adv_audio_out, waveform_plot],
    )

    gr.Markdown(
        "> **Tip:** Using a microphone on macOS requires mic access in "
        "*System Settings → Privacy & Security → Microphone*."
    )

demo_app.launch(share=False, inbrowser=True, quiet=False)


/var/folders/0f/glx6yb212vlc0p67p0phsp180000gn/T/ipykernel_39917/4125422845.py:12: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Step 7: Live Streaming Transcript Demo

A real-time alternative interface that continuously transcribes microphone audio as you speak, while keeping history aligned to natural utterance boundaries.

| Feature | Detail |
|---|---|
| Live preview | Re-transcribes the current utterance every ~1 s for smoother continuity |
| History logging | Commits only finalized utterances (silence-based) or forced long segments |
| Overlap handling | Keeps short carry-over audio and removes repeated boundary words in history |
| Stream normalization | Handles cumulative Gradio tuple payloads by extracting only new audio |
| Attack modes | Same three modes as Step 6 (Clean / UAP / CW Injection) |

> Uses Gradio's `Audio(streaming=True)` + `.stream()` event.  
> Chrome/Firefox work best; Safari may block microphone streams.

In [6]:
# -- Streaming state & inference ---------------------------------------------

import re
import time
import numpy as np
import torch
import resampy

LIVE_STEP_S             = 1.0   # Minimum new audio before refreshing live preview
COMMIT_SILENCE_S        = 1.2   # Consecutive silence required to finalize an utterance
MAX_SEGMENT_S           = 14.0  # Force a history commit for very long uninterrupted speech
MIN_SPEECH_S            = 0.3   # Ignore very short bursts that are likely noise
CARRYOVER_S             = 1.0   # Preserve a short tail when a long utterance is split
ASSUME_TUPLE_CUMULATIVE = True  # Gradio tuple payloads are often cumulative since stream start
SR                      = 16000
SILENCE_THRESHOLD       = 0.01


def _samples(seconds: float) -> int:
    return int(seconds * SR)


def _empty_audio() -> np.ndarray:
    return np.zeros(0, dtype=np.float32)


def _init_stream_state():
    return {
        "segment_chunks": [],
        "samples_since_live": 0,
        "speech_samples": 0,
        "silence_samples": 0,
        "last_tuple_len": 0,
        "last_tuple_audio": None,
        "last_activity_ts": time.monotonic(),
        "live_clean": "-",
        "live_adv": "-",
        "live_status": "Waiting for audio...",
        "history_clean": "",
        "history_adv": "",
    }


def _decode_chunk(chunk):
    """Convert Gradio audio payloads to a mono float32 waveform at 16 kHz."""
    if chunk is None:
        return None
    if isinstance(chunk, tuple):
        sr_in, data = chunk
        if data is None or (hasattr(data, "__len__") and len(data) == 0):
            return None
        data = np.asarray(data, dtype=np.float32)
        if data.ndim == 2:
            data = data.mean(axis=1)
        if data.size and np.max(np.abs(data)) > 1.5:
            data = data / 32768.0
        if sr_in and sr_in != SR:
            data = resampy.resample(data, sr_in, SR)
        return np.clip(data, -1.0, 1.0)
    if isinstance(chunk, (bytes, bytearray)):
        pcm = np.frombuffer(chunk, dtype=np.int16).astype(np.float32) / 32768.0
        return np.clip(pcm, -1.0, 1.0)
    return None


def _extract_new_audio(chunk, decoded_audio: np.ndarray, state) -> np.ndarray:
    """
    Normalize Gradio streaming input to only the NEW part since last callback.
    Handles both cumulative tuple payloads and fixed-size incremental chunks.
    """
    if decoded_audio is None or len(decoded_audio) == 0:
        return _empty_audio()

    if isinstance(chunk, tuple) and ASSUME_TUPLE_CUMULATIVE:
        cur_len = len(decoded_audio)
        prev_len = int(state.get("last_tuple_len", 0))
        prev_audio = state.get("last_tuple_audio")

        if prev_len <= 0:
            state["last_tuple_len"] = cur_len
            state["last_tuple_audio"] = decoded_audio.copy()
            return decoded_audio

        if cur_len == prev_len:
            # If payload bytes are identical, nothing new arrived. If same length but
            # different samples, treat it as incremental (non-cumulative) chunks.
            if (
                isinstance(prev_audio, np.ndarray)
                and len(prev_audio) == cur_len
                and np.array_equal(decoded_audio, prev_audio)
            ):
                return _empty_audio()
            state["last_tuple_len"] = cur_len
            state["last_tuple_audio"] = decoded_audio.copy()
            return decoded_audio

        if cur_len > prev_len:
            state["last_tuple_len"] = cur_len
            state["last_tuple_audio"] = decoded_audio.copy()
            return decoded_audio[prev_len:]

        # Stream restarted (new recording/session or browser reset).
        state["last_tuple_len"] = cur_len
        state["last_tuple_audio"] = decoded_audio.copy()
        return decoded_audio

    return decoded_audio


def _concat_chunks(chunks) -> np.ndarray:
    return np.concatenate(chunks) if chunks else _empty_audio()


def _append_chunk(chunks, chunk: np.ndarray, max_samples: int | None = None):
    chunks.append(chunk)
    if max_samples is None:
        return

    total = sum(len(part) for part in chunks)
    while total > max_samples and chunks:
        overflow = total - max_samples
        head = chunks[0]
        if overflow >= len(head):
            total -= len(head)
            chunks.pop(0)
            continue
        chunks[0] = head[overflow:]
        total -= overflow


def _is_silent(audio: np.ndarray, threshold: float = SILENCE_THRESHOLD) -> bool:
    if len(audio) == 0:
        return True
    rms = float(np.sqrt(np.mean(audio ** 2)))
    return rms < threshold


def _trim_silence_edges(audio: np.ndarray, frame_s: float = 0.1) -> np.ndarray:
    if len(audio) == 0:
        return audio

    frame = max(1, _samples(frame_s))
    start = 0
    end = len(audio)

    while start + frame < end and _is_silent(audio[start:start + frame]):
        start += frame
    while end - frame > start and _is_silent(audio[end - frame:end]):
        end -= frame

    return audio[start:end]


def _word_key(word: str) -> str:
    return re.sub(r"\W+", "", word).lower()


def _overlap_word_count(previous_text: str, new_text: str, max_words: int = 16) -> int:
    previous_words = previous_text.strip().split()
    new_words = new_text.strip().split()
    max_overlap = min(max_words, len(previous_words), len(new_words))

    for size in range(max_overlap, 0, -1):
        prev_slice = [_word_key(word) for word in previous_words[-size:]]
        new_slice = [_word_key(word) for word in new_words[:size]]
        if prev_slice == new_slice and all(prev_slice):
            return size
    return 0


def _append_history(history_text: str, new_text: str) -> str:
    compact_text = " ".join(new_text.split())
    if not compact_text:
        return history_text

    lines = [line for line in history_text.splitlines() if line.strip()]
    previous_line = lines[-1] if lines else ""
    overlap = _overlap_word_count(previous_line, compact_text)
    merged_words = compact_text.split()[overlap:]
    if not merged_words:
        return history_text
    return history_text + " ".join(merged_words) + "\n"


def _transcribe_with_mode(audio: np.ndarray, stream_mode: str):
    clean_text = model.transcribe(torch.from_numpy(audio))

    if stream_mode == "Clean":
        return clean_text, clean_text, "Clean - no attack"

    pert = ATTACK_MODES.get(stream_mode)
    if pert is None:
        return clean_text, clean_text, f"Unknown mode: {stream_mode}"

    adv_audio = apply_perturbation(audio, pert)
    adv_text = model.transcribe(torch.from_numpy(adv_audio))

    if stream_mode == "Untargeted UAP":
        status = "Untargeted UAP applied"
    else:
        hit = TARGET_PHRASE.lower() in adv_text.lower()
        status = f"{'[OK] Injected' if hit else '[--] Not injected'} - target: \"{TARGET_PHRASE}\""

    return clean_text, adv_text, status


def _reset_segment(state, carryover_audio: np.ndarray | None = None):
    carryover_audio = _empty_audio() if carryover_audio is None else carryover_audio.astype(np.float32)
    state["segment_chunks"] = [carryover_audio] if len(carryover_audio) else []
    state["samples_since_live"] = 0
    state["speech_samples"] = len(carryover_audio)
    state["silence_samples"] = 0


def _stream_outputs(state):
    return (
        state,
        state["live_clean"],
        state["live_adv"],
        state["live_status"],
        state["history_clean"],
        state["history_adv"],
    )


def _commit_segment(state, stream_mode: str, keep_tail: bool):
    segment_audio = _trim_silence_edges(_concat_chunks(state["segment_chunks"]))
    if len(segment_audio) == 0:
        _reset_segment(state)
        state["live_status"] = "Silence - waiting for speech..."
        return _stream_outputs(state)

    clean_text, adv_text, status = _transcribe_with_mode(segment_audio, stream_mode)
    state["live_clean"] = clean_text or state["live_clean"]
    state["live_adv"] = adv_text or state["live_adv"]
    state["live_status"] = status
    state["history_clean"] = _append_history(state["history_clean"], clean_text)
    state["history_adv"] = _append_history(state["history_adv"], adv_text)

    carryover_audio = None
    if keep_tail and len(segment_audio) > _samples(CARRYOVER_S):
        carryover_audio = segment_audio[-_samples(CARRYOVER_S):].copy()
    _reset_segment(state, carryover_audio=carryover_audio)
    return _stream_outputs(state)


def finalize_stream(state, stream_mode):
    """Force-commit pending utterance when microphone streaming stops."""
    state = _init_stream_state() if state is None else state
    if state["speech_samples"] < _samples(MIN_SPEECH_S):
        state["live_status"] = "Stopped - no utterance to commit."
        return _stream_outputs(state)
    return _commit_segment(state, stream_mode=stream_mode, keep_tail=False)


def stream_transcribe(new_chunk, state, stream_mode):
    state = _init_stream_state() if state is None else state
    now = time.monotonic()

    decoded = _decode_chunk(new_chunk)
    chunk_np = _extract_new_audio(new_chunk, decoded, state)
    if len(chunk_np) == 0:
        # Some browsers emit periodic callbacks with no new samples.
        # Commit on idle silence if we already have enough speech buffered.
        idle_s = now - float(state.get("last_activity_ts", now))
        if state["speech_samples"] >= _samples(MIN_SPEECH_S) and idle_s >= COMMIT_SILENCE_S:
            return _commit_segment(state, stream_mode=stream_mode, keep_tail=False)
        return _stream_outputs(state)

    state["last_activity_ts"] = now
    _append_chunk(state["segment_chunks"], chunk_np, max_samples=_samples(MAX_SEGMENT_S + CARRYOVER_S))
    state["samples_since_live"] += len(chunk_np)

    if _is_silent(chunk_np):
        state["silence_samples"] += len(chunk_np)
    else:
        state["speech_samples"] += len(chunk_np)
        state["silence_samples"] = 0

    if state["speech_samples"] < _samples(MIN_SPEECH_S):
        if state["silence_samples"] >= _samples(COMMIT_SILENCE_S):
            _reset_segment(state)
            state["live_status"] = "Silence - waiting for speech..."
        else:
            state["live_status"] = "Listening..."
        return _stream_outputs(state)

    segment_audio = _concat_chunks(state["segment_chunks"])
    should_commit_for_silence = state["silence_samples"] >= _samples(COMMIT_SILENCE_S)
    should_commit_for_length = len(segment_audio) >= _samples(MAX_SEGMENT_S)

    if should_commit_for_silence or should_commit_for_length:
        return _commit_segment(
            state,
            stream_mode=stream_mode,
            keep_tail=should_commit_for_length and not should_commit_for_silence,
        )

    if state["samples_since_live"] < _samples(LIVE_STEP_S):
        state["live_status"] = "Listening..."
        return _stream_outputs(state)

    # Live preview transcribes the current utterance segment for continuity.
    state["samples_since_live"] = 0
    preview_audio = _trim_silence_edges(segment_audio)
    if len(preview_audio) == 0:
        state["live_status"] = "Listening..."
        return _stream_outputs(state)

    clean_text, adv_text, status = _transcribe_with_mode(preview_audio, stream_mode)
    state["live_clean"] = clean_text or state["live_clean"]
    state["live_adv"] = adv_text or state["live_adv"]
    state["live_status"] = status
    return _stream_outputs(state)

In [7]:
# -- Streaming Gradio UI -----------------------------------------------------

STREAM_SUBTITLE = (
    "**Speak into your microphone** - live transcripts refresh automatically while the history log is committed on utterance boundaries.\n\n"
    "| Mode | Description |\n"
    "|---|---|\n"
    "| **Clean** | Baseline Whisper - no modification |\n"
    "| **Untargeted UAP** | Imperceptible noise that degrades accuracy |\n"
    f"| **Targeted CW Injection** | Forces Whisper to output *\"{TARGET_PHRASE}\"* |\n"
)

with gr.Blocks(
    title="SoundFinal - Live Stream Demo (AAI-590)",
    theme=gr.themes.Soft(primary_hue="indigo"),
) as live_demo:

    gr.Markdown("# SoundFinal - Live Streaming Transcript  \n### AAI-590 Final Project")

    stream_state = gr.State(None)

    with gr.Row():
        with gr.Column(scale=1, min_width=280):
            gr.Markdown(STREAM_SUBTITLE)

            mic_in = gr.Audio(
                sources=["microphone"],
                type="numpy",
                streaming=True,
                label="Microphone (streams automatically)",
            )
            stream_mode_sel = gr.Radio(
                choices=["Clean"] + list(ATTACK_MODES.keys()),
                value="Clean",
                label="Attack Mode",
            )
            gr.Markdown(
                "_Change the attack mode at any time - the next live refresh will use the new mode._"
            )

        with gr.Column(scale=2):
            gr.Markdown("### Live Transcripts")
            with gr.Row():
                live_clean_box = gr.Textbox(
                    label="Clean Transcript (latest live window)",
                    interactive=False,
                    lines=3,
                    placeholder="Transcript will appear here...",
                )
                live_adv_box = gr.Textbox(
                    label="Adversarial Transcript (latest live window)",
                    interactive=False,
                    lines=3,
                    placeholder="Adversarial transcript will appear here...",
                )
            live_status_box = gr.Textbox(
                label="Status",
                interactive=False,
                lines=1,
                value="Waiting for audio...",
            )
            gr.Markdown("### Session Transcript Log")
            with gr.Row():
                history_clean_box = gr.Textbox(
                    label="Committed history clean",
                    interactive=False,
                    lines=12,
                    max_lines=40,
                    autoscroll=True,
                    placeholder="Finalized clean utterances will be logged here...",
                )
                history_adv_box = gr.Textbox(
                    label="Committed history adversarial",
                    interactive=False,
                    lines=12,
                    max_lines=40,
                    autoscroll=True,
                    placeholder="Finalized adversarial utterances will be logged here...",
                )

    mic_in.stream(
        fn=stream_transcribe,
        inputs=[mic_in, stream_state, stream_mode_sel],
        outputs=[stream_state, live_clean_box, live_adv_box, live_status_box, history_clean_box, history_adv_box],
    )

    # Flush pending utterance at end of recording so the history log is updated.
    mic_in.stop_recording(
        fn=finalize_stream,
        inputs=[stream_state, stream_mode_sel],
        outputs=[stream_state, live_clean_box, live_adv_box, live_status_box, history_clean_box, history_adv_box],
    )

    gr.Markdown(
        "> **Tip:** Allow microphone access when prompted.  \n"
        "> Best results with a headset to reduce room echo picked up by Whisper."
    )

live_demo.launch(share=False, inbrowser=True, quiet=False)

/var/folders/0f/glx6yb212vlc0p67p0phsp180000gn/T/ipykernel_39917/1671028471.py:12: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


/opt/homebrew/anaconda3/envs/capstone/lib/python3.11/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [1, 3001, 201]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  return _VF.stft(  # type: ignore[attr-defined]
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it w

## Streaming Refactor Notes

### What changed

The streaming callback now separates two responsibilities that were previously mixed together:

1. A short live context buffer drives the on-screen transcript refresh every ~1 second.
2. A longer utterance buffer is used only for committed history entries.

This removes the old coupling where the same rolling window was used both for real-time feedback and for the permanent transcript log.

### How the new history logic works

- Silence is treated as an utterance boundary only after a sustained pause (`COMMIT_SILENCE_S`), instead of reacting to a single quiet chunk.
- Very short bursts are ignored (`MIN_SPEECH_S`) so background noise does not create empty or partial history entries.
- If someone speaks for too long without pausing, the code forces a commit (`MAX_SEGMENT_S`) and keeps a short audio tail (`CARRYOVER_S`) for the next segment.
- When that carry-over causes repeated words at the text boundary, the append step removes the overlapping prefix before adding the new line to history.

### Why this fixes the missing and overlapping transcript issues

The live boxes still update from a rolling recent window, so responsiveness stays high. The history log, however, is written only from finalized utterances or controlled long-segment splits. That means:

- fewer missing words caused by committing an unstable intermediate window,
- fewer duplicated phrases caused by overlapping audio windows,
- clearer state management, because live preview and committed history no longer fight over the same buffer.

## Troubleshooting Tips

1. **Attack fails (target phrase not found)**: Increase `CW_C` (e.g., `100.0`) or `CW_STEPS` (e.g., `1000`). A larger `c` puts more weight on the adversarial loss relative to the L2 penalty.
2. **Low SNR / audible artifacts**: Reduce `CW_C` — the binary search (`CW_BS_STEPS`) will find a tighter `c` automatically. Also ensure `CW_STEPS >= 500`.
3. **Out of memory (MPS/CUDA)**: Reduce to `model_path="openai/whisper-tiny"`. All `WhisperASRWithAttack` calls support any whisper variant.
4. **Recording is silent / near-zero**: Ensure your microphone is permitted in macOS System Settings → Privacy → Microphone. Check that `sounddevice` lists your expected device with `import sounddevice as sd; print(sd.query_devices())`.
5. **`VOICE_FILE` missing on model step**: Run Step 1 (recording cell) first, or place a 16 kHz WAV at `demo_assets/my_voice_generic.wav` manually.
